# Week 13 Lab: Unsupervised Learning (Clustering and PCA)

<a href="https://colab.research.google.com/github/bradleyboehmke/uc-bana-4080/blob/main/labs/lab-13-clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this lab, there is no target `y` to predict. You'll use K-Means to find groups of similar customers and turn them into named segments, use PCA to compress many correlated columns into a few components, and then combine the two to see whether the structure PCA finds matches labels it never saw.

## 🎯 Learning Objectives
By the end of this lab, you will be able to:
- Scale features and fit K-Means, then use inertia (the elbow) and silhouette scores to choose the number of clusters
- Profile clusters and turn them into named, actionable segments
- Run the PCA workflow: standardize, fit, read a scree plot, and interpret loadings
- Use PCA components as model inputs, and judge what that costs and saves
- Cluster on PCA components and compare the clusters with known labels

## 📚 This Lab Reinforces
- **Chapter 31: Unsupervised Learning and Clustering**
- **Chapter 32: Dimension Reduction with PCA**
- **Week 13 Tuesday Slides: Unsupervised Learning**

## 🕐 Estimated Time & Structure
**Total Time:** 75 minutes
**Mode:** Work together in groups (2–4 students), submit individually

- **[0–25 min]** Guided review: Tuesday's customers and wine data
- **[25–30 min]** Class discussion and Q&A
- **[30–70 min]** Independent challenges with the Ames housing and breast tumor data
- **[70–75 min]** Lab wrap-up and homework quiz preparation

You are encouraged to work through the lab with **2–4 classmates**: talk through the problems and help each other get past coding hurdles. However, **you must write your own code, and you will submit your own notebook and homework quiz.**

## 💡 Why This Matters
Most business data has no label telling you what kind of customer, product or transaction you're looking at. Clustering turns that data into segments a team can act on, and PCA makes wide, redundant data small enough to see and to model. Both need judgment, because there is no accuracy score to tell you when you're right.

## Setup

We'll work with four datasets:

- **Customers** and **wine** for the guided review: the same data as Tuesday's lecture
- **Ames housing** and **breast tumor measurements** for the independent challenges (loaded in Part B)

Run this cell to import everything and set the shared `RANDOM_STATE`.

In [ ]:
# Required imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import silhouette_score, accuracy_score, homogeneity_score, completeness_score

# One random state for every split, KMeans, PCA, and model
RANDOM_STATE = 42

print("✓ Libraries imported successfully!")

---

## Part A — Guided Review: Tuesday's Workflow (25 minutes)

Let's rerun Tuesday's two workflows on the lecture's data.

**📋 Workflow:**
1. Cluster the customers with K-Means, then experiment with the number of clusters
2. Use the elbow and silhouette scores to choose K
3. Profile the clusters and name them as segments
4. Run PCA on the wine data and read the scree plot and loadings
5. Draw the 13 wine measurements in 2-D, then see what happens without scaling

Run each cell, and when you see 🔧 **Try it**, change the values and re-run the cell.

### Step 1: The Customers

The same simulated customers as Tuesday: each one's annual spending and number of store visits per year. There's no column saying which "type" of customer anyone is.

In [ ]:
# Step 1: Simulated customers (same as Tuesday's lecture)
rng = np.random.default_rng(42)
customers = pd.DataFrame({
    "annual_spending": np.concatenate([rng.normal(2000, 500, 30), rng.normal(5000, 800, 30), rng.normal(8500, 700, 30)]),
    "visits_per_year": np.concatenate([rng.normal(5, 2, 30), rng.normal(15, 3, 30), rng.normal(35, 5, 30)]),
})

plt.figure(figsize=(7, 5))
plt.scatter(customers["annual_spending"], customers["visits_per_year"], color="gray", edgecolor="white")
plt.xlabel("Annual spending ($)")
plt.ylabel("Visits per year")
plt.title("90 customers, no labels")
plt.show()

### Step 2: Scale, Then Fit K-Means

K-Means measures distance, so we scale first. Otherwise spending (thousands of dollars) would drown out visits (a few dozen).

In [ ]:
# Step 2: Scale, fit K-Means, and score the clusters
customers_scaled = StandardScaler().fit_transform(customers)

n_clusters = 3   # 🔧 change this value
kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=RANDOM_STATE)
cluster_labels = kmeans.fit_predict(customers_scaled)

print(f"K = {n_clusters}: inertia {kmeans.inertia_:.1f}, silhouette {silhouette_score(customers_scaled, cluster_labels):.3f}")

plt.figure(figsize=(7, 5))
plt.scatter(customers["annual_spending"], customers["visits_per_year"], c=cluster_labels, cmap="tab10", edgecolor="white")
plt.xlabel("Annual spending ($)")
plt.ylabel("Visits per year")
plt.title(f"K-Means with K = {n_clusters}")
plt.show()

🔧 **Try it (2–3 minutes):** Re-run the cell above with `n_clusters` set to **2, 4 and 6**. Note the inertia and silhouette each time.

- What happens to **inertia** every time you add a cluster? Why can't "lowest inertia" be the rule for choosing K?
- At K = 4 and 6, which real groups did K-Means split? What happened to the silhouette?

**Set `n_clusters` back to 3 and re-run the cell before moving on.**

*Your notes:*

### Step 3: Choose K with the Elbow and Silhouette

Instead of trying K values one at a time, loop over them and plot both scores, just like the slider on Tuesday.

In [ ]:
# Step 3: Inertia and silhouette for K = 2 to 8
k_values = range(2, 9)
inertias, silhouettes = [], []
for k in k_values:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(customers_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(customers_scaled, km.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(k_values, inertias, marker="o", color="#399de5")
ax1.set(title="Elbow: inertia by K", xlabel="K", ylabel="Inertia (lower = tighter)")
ax2.plot(k_values, silhouettes, marker="o", color="#e58139")
ax2.set(title="Silhouette score by K", xlabel="K", ylabel="Silhouette (higher = better separated)")
plt.tight_layout()
plt.show()

**Reading the plots:** look for the **elbow** (where inertia stops dropping sharply) and the **peak** silhouette. On this clean data they agree. On real data they often don't, and then the business decides between the options the scores leave open.

### 🧪 Practice Exercise — From Clusters to Segments

**Business Scenario:** The marketing team can't use "cluster 0, 1 and 2". They need to know who is in each group and what to do about them.

**Your Task:**

1. Using `cluster_labels` from Step 2 (with `n_clusters = 3`), compute the **average spending and visits per cluster**, plus how many customers each cluster has.
2. Give each cluster a descriptive **segment name**, like Tuesday's "Occasional shoppers".
3. Suggest **one marketing action** for each segment.

💡 **Hint:** `customers.assign(cluster=cluster_labels).groupby("cluster").agg(...)` with named aggregations such as `avg_spending=("annual_spending", "mean")`. Sort by spending, because the cluster numbers themselves are arbitrary.

In [ ]:
# Your code here

*Steps 2–3 — Your segment names and marketing actions:*

### Step 4: PCA on Wine

**Wine:** 178 wines, 13 chemical measurements each, from three cultivars (grape varieties). We'll use the cultivar only to *check* PCA's results, never to fit PCA.

Scale, fit PCA with all 13 components, and look at the **scree plot**: each component's share of the variance (bars) and the running total (line).

In [ ]:
# Step 4: Scale the wine data, fit PCA, and draw the scree plot
wine = load_wine(as_frame=True)
X_wine, y_wine = wine.data, wine.target

wine_scaled = StandardScaler().fit_transform(X_wine)
pca_wine = PCA().fit(wine_scaled)

var_pct = 100 * pca_wine.explained_variance_ratio_
cum_pct = var_pct.cumsum()
components = range(1, len(var_pct) + 1)

plt.figure(figsize=(9, 4))
plt.bar(components, var_pct, color="#e58139", label="Each component")
plt.plot(components, cum_pct, marker="o", color="#399de5", label="Running total")
plt.axhline(90, color="gray", linestyle="--", label="90%")
plt.xlabel("Principal component")
plt.ylabel("Variance explained (%)")
plt.xticks(list(components))
plt.legend()
plt.show()

print(f"PC1 alone: {var_pct[0]:.1f}%  |  PC1 + PC2: {cum_pct[1]:.1f}%  |  components for 90%: {(cum_pct >= 90).argmax() + 1}")

### Step 5: What Do PC1 and PC2 Mean?

Each component is a weighted mix of the 13 original columns. Those weights are its **loadings**. Large weights, positive or negative, tell you which columns move together in that component.

In [ ]:
# Step 5: Loadings for the first two components
loadings = pd.DataFrame(pca_wine.components_[:2].T, index=X_wine.columns, columns=["PC1", "PC2"])
loadings.sort_values("PC1").round(2)

**Reading the loadings:** PC1's biggest positive weights are flavanoids, total phenols and related measures, so PC1 roughly measures "phenolic richness". The sign is arbitrary: a different run could flip every weight, and the pattern would mean the same thing.

### Step 6: 13 Columns, One Picture

Keep just 2 components and plot every wine. The colors are the true cultivars, added only after PCA is done.

In [ ]:
# Step 6: A 2-D view of the wine data
use_scaling = True   # 🔧 try False

wine_input = StandardScaler().fit_transform(X_wine) if use_scaling else X_wine
pca_2 = PCA(n_components=2)
wine_2d = pca_2.fit_transform(wine_input)

plt.figure(figsize=(7, 5))
for cultivar, color in zip(range(3), ["#e58139", "#399de5", "#59a14f"]):
    mask = (y_wine == cultivar).to_numpy()
    plt.scatter(wine_2d[mask, 0], wine_2d[mask, 1], color=color, label=f"Cultivar {cultivar}", edgecolor="white")
plt.xlabel(f"PC1 ({pca_2.explained_variance_ratio_[0]:.1%} of variance)")
plt.ylabel(f"PC2 ({pca_2.explained_variance_ratio_[1]:.1%} of variance)")
plt.title("Scaled" if use_scaling else "NOT scaled")
plt.legend()
plt.show()

🔧 **Try it (2–3 minutes):** Set `use_scaling = False` and re-run the cell above.

- What happens to PC1's share of the variance? Does that make PC1 better?
- How well are the three cultivars separated now?
- Run `X_wine.std().sort_values(ascending=False).head()` in a new cell. Which column is PC1 really measuring?

*Your notes:*

## Class Discussion/Q&A (5 minutes)

**Discussion prompts:**
- Inertia always went down as you added clusters. In your own words, why do we look for an elbow instead of the lowest inertia?
- Your segment names came from you, not from K-Means. What would you need to know about a business before choosing between 3 and 4 segments?
- Without scaling, PC1 explained almost all of the wine variance, yet the 2-D picture got worse. How can "more variance explained" be worse?

**Common blockers and clarifications:**
- Scale before K-Means **and** before PCA: both are driven by distances and variances
- Cluster numbers are arbitrary names: cluster 0 in one run can be cluster 2 in another, so always profile before you compare
- `n_init=10` and `random_state` make K-Means repeatable. Without them, your clusters may not match your groupmates'
- PCA never sees `y`: the cultivar colors were added afterwards, only to check the result

---

## Part B — Independent Challenges (40 minutes)

Now it's your turn to apply K-Means and PCA to two real datasets: the Ames housing data and a set of breast tumor measurements. The specific numerical results you get will be used to answer questions on this week's **Canvas homework quiz**.

* **The support decreases as you go.** Challenges 1–2 give you starter code with blanks (🧩), Challenge 3 gives you an outline only (📝), and Challenge 4 is on your own (🚀).
* **Work together, submit individually.** Talk through the challenges with your group and help each other get unstuck, but **write your own code**. You'll each submit your own notebook and quiz.
* **DO NOT USE AI** to generate code for you.
* Feel free to ask questions or seek help from the instructor. Periodically, we'll stop and walk through parts of a challenge together.
* **Important:** use the exact settings given: `random_state=RANDOM_STATE` in every split, `KMeans`, `PCA` and `LogisticRegression`, `n_init=10` for every `KMeans`, and `test_size=0.2, stratify=y` for the split. These produce the exact numbers needed for the quiz. Steps marked **📝** feed directly into quiz questions.

### Challenge 1 — Segment the Ames Housing Market

🧩 **Starter code provided:** fill in the `___` blanks.

**Business Context:** You're a real estate analyst. Your marketing team wants to split the Ames housing market into **3 segments** so they can tailor their listings and ads to each one.

**Your Task:**
1. **Prepare the data**: Use these six features: `GrLivArea`, `TotalBsmtSF`, `GarageArea`, `YearBuilt`, `OverallQual`, `OverallCond`. Fill any missing `TotalBsmtSF` and `GarageArea` values with 0 (no basement or no garage), then standardize all six.
2. **Choose K**: Fit `KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)` for every k from 2 to 10, recording inertia and silhouette score. 📝 Which k has the highest silhouette score?
3. **Fit the final model** with k=3 (the number of segments marketing asked for), using the same settings. 📝 Record the silhouette score (4 decimals), the inertia (2 decimals), and the number of homes in each cluster.
4. **Profile the clusters**: Compare the average feature values in each cluster, then give each one a segment name a marketer would understand.

In [ ]:
# Step 1: Load the data (PROVIDED), select the features, fill missing values, and standardize
ames = pd.read_csv('https://raw.githubusercontent.com/bradleyboehmke/uc-bana-4080/main/data/ames_clean.csv')
features = ['GrLivArea', 'TotalBsmtSF', 'GarageArea', 'YearBuilt', 'OverallQual', 'OverallCond']

ames_cluster = ames[features].copy()
ames_cluster['TotalBsmtSF'] = ames_cluster['TotalBsmtSF'].fillna(___)
ames_cluster['GarageArea'] = ames_cluster['GarageArea'].fillna(___)

scaler = StandardScaler()
ames_scaled = scaler.___(ames_cluster)
print(f"{ames_scaled.shape[0]} homes, {ames_scaled.shape[1]} features")

In [ ]:
# Step 2: Fit K-Means for every k from 2 to 10, recording inertia and silhouette score
k_values = range(___, ___)
inertias = []
silhouettes = []
for k in k_values:
    kmeans = KMeans(n_clusters=___, n_init=10, random_state=RANDOM_STATE)
    kmeans.fit(___)
    inertias.append(kmeans.___)
    silhouettes.append(silhouette_score(ames_scaled, kmeans.___))

# Plot the elbow and silhouette curves (PROVIDED)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(list(k_values), inertias, marker='o')
ax1.set(xlabel='Number of clusters (k)', ylabel='Inertia', title='Elbow method')
ax2.plot(list(k_values), silhouettes, marker='o', color='darkorange')
ax2.set(xlabel='Number of clusters (k)', ylabel='Silhouette score', title='Silhouette scores')
plt.tight_layout()
plt.show()

best_k = list(k_values)[np.argmax(silhouettes)]
print(f"Highest silhouette score: k={best_k} 📝")

In [ ]:
# Step 3: Fit the final model with k=3
kmeans_final = KMeans(n_clusters=___, n_init=10, random_state=RANDOM_STATE)
ames_cluster['Cluster'] = kmeans_final.fit_predict(___)

print(f"Silhouette score: {silhouette_score(ames_scaled, ames_cluster['Cluster']):.4f} 📝")
print(f"Inertia: {kmeans_final.___:.2f} 📝")
print("\nHomes per cluster 📝")
print(ames_cluster['Cluster'].___().sort_index())

In [ ]:
# Step 4: Profile each cluster by its average feature values
ames_cluster.groupby(___)[features].___().round(1)

*Step 4 — Your segment names (one per cluster, with a sentence on what makes it different):*

### Challenge 2 — Compress 30 Tumor Measurements with PCA

🧩 **Starter code provided:** fill in the `___` blanks.

**Business Context:** You're working with medical researchers who recorded **30 measurements** of each breast tumor (radius, texture, concavity, and so on). Many of the measurements are closely related, which makes the data hard to visualize and explore. You'll use PCA to see how much of the information a few components can hold.

**Your Task:**
1. **Prepare the data**: Separate the 30 measurement columns (`X`) from the `diagnosis` column (`y`, M = malignant, B = benign), then standardize `X`. This challenge explores all 569 patients, so there is no train/test split yet.
2. **Fit PCA** with all components: `PCA()`. 📝 Record the percentage of variance explained by PC1, and by PC1 and PC2 together (2 decimals).
3. **Count components**: 📝 How many components are needed to explain at least 90% of the variance? At least 95%?
4. **Read the loadings**: 📝 Which 5 features have the largest **absolute** loadings on PC1? What do they have in common?
5. **Visualize**: Run the provided scree plot.

In [ ]:
# Step 1: Load the data (PROVIDED), separate the features from the diagnosis, and standardize
breast_cancer = pd.read_csv('https://raw.githubusercontent.com/bradleyboehmke/uc-bana-4080/main/data/breast_cancer.csv')
X = breast_cancer.drop(columns='diagnosis')
y = breast_cancer['diagnosis']

scaler = StandardScaler()
X_scaled = scaler.___(___)
print(f"{X.shape[1]} features, {X.shape[0]} patients ({(y == 'M').sum()} malignant, {(y == 'B').sum()} benign)")

In [ ]:
# Step 2: Fit PCA with all components and compute the share of variance each one explains
pca = PCA()
pca.fit(___)
explained = pca.___               # share of the variance for each component
cumulative = np.___(explained)    # running total

print(f"PC1 explains {explained[0] * 100:.2f}% of the variance 📝")
print(f"PC1 + PC2 explain {cumulative[___] * 100:.2f}% 📝")

# Step 3: How many components to reach at least 90% and at least 95% of the variance?
n_90 = np.argmax(cumulative >= ___) + 1
n_95 = np.argmax(cumulative >= ___) + 1
print(f"Components for 90%: {n_90} 📝")
print(f"Components for 95%: {n_95} 📝")

In [ ]:
# Step 4: Top 5 features by absolute PC1 loading
loadings = pd.DataFrame(
    pca.components_.T,
    index=X.columns,
    columns=[f'PC{i + 1}' for i in range(pca.n_components_)]
)
top5 = loadings['PC1'].___().nlargest(___)
print("Top 5 features on PC1 (absolute loading) 📝")
print(top5.round(3))

*Step 4 — What do PC1's top features have in common? What might a high PC1 score mean for a tumor?*

In [ ]:
# Step 5: Scree plot (PROVIDED)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
components = range(1, len(explained) + 1)
ax1.bar(components, explained)
ax1.set(xlabel='Principal component', ylabel='Share of variance', title='Scree plot')
ax2.plot(components, cumulative, marker='o')
ax2.axhline(0.90, color='darkorange', linestyle='--', label='90%')
ax2.axhline(0.95, color='green', linestyle='--', label='95%')
ax2.set(xlabel='Number of components', ylabel='Cumulative share', title='Cumulative variance explained')
ax2.legend()
plt.tight_layout()
plt.show()

### Challenge 3 — Feed PCA Components to a Classifier

📝 **Outline only:** the code cell lists each step as a comment. You write all the code.

**Business Context:** The researchers want a model that classifies tumors as malignant or benign. A model with fewer inputs is faster and easier to maintain, so they ask: *can PCA components replace the 30 original measurements without hurting accuracy?*

**Your Task:**
1. **Split and scale**: Split `X` and `y` with `test_size=0.2`, `random_state=RANDOM_STATE`, `stratify=y`. Fit a `StandardScaler` on the **training** data only, then transform both the training and test data.
2. **Baseline (all 30 features)**: Train `LogisticRegression(max_iter=10000, random_state=RANDOM_STATE)` on the scaled training data. 📝 Record the test accuracy and the overfitting gap (training accuracy minus test accuracy), both to 4 decimals.
3. **PCA model (95% of the variance)**: Fit `PCA(n_components=0.95, random_state=RANDOM_STATE)` on the scaled **training** data, transform both sets, and train the same logistic regression on the components. 📝 Record how many components were kept, the test accuracy, the overfitting gap, and the dimensionality reduction: `(1 - components / 30) * 100`.
4. **Compare component counts**: Repeat Step 3 with `PCA(n_components=n, random_state=RANDOM_STATE)` for each `n` in `[2, 5, 10, 15, 20, 25]`, recording training and test accuracy. 📝 Which number of components gives the highest test accuracy?
5. **Visualize and reflect**: Plot test accuracy against the number of components, with a horizontal line at the baseline's test accuracy.

💡 **Hint:** PCA is fit like a scaler: `fit_transform()` on the training data, then `transform()` on the test data. Fitting it on the test data leaks information.

In [ ]:
# Step 1: Split X and y (test_size=0.2, random_state=RANDOM_STATE, stratify=y),
#         then fit a StandardScaler on the training data and transform both sets

# Step 2: Baseline logistic regression on all 30 scaled features
#         (max_iter=10000, random_state=RANDOM_STATE); print test accuracy and gap 📝

# Step 3: PCA(n_components=0.95, random_state=RANDOM_STATE): fit on scaled training data,
#         transform both; train the same model; print components kept, test accuracy,
#         gap, and dimensionality reduction 📝

# Step 4: Loop over n in [2, 5, 10, 15, 20, 25]; for each, fit PCA and the model,
#         and record training and test accuracy; print which n has the highest test accuracy 📝

# Step 5: Plot test accuracy vs. number of components, with the baseline as a horizontal line


*Step 5 — Your answers:*

1. Did the 95%-variance PCA model beat the baseline on the test set? Compare the two overfitting gaps. Why might fewer, uncorrelated inputs generalize better?
2. In Step 4 you compared six component counts **on the test set** and picked the best. Based on Week 12, why is that best test accuracy likely optimistic? How would you choose the number of components properly?

### Challenge 4 — Find the Diagnoses Without Labels

🚀 **On your own:** no starter code. Use `X_scaled` and `y` from Challenge 2.

**Business Context:** The researchers ask a bold question: *if we had never recorded a single diagnosis, would the tumor measurements alone separate malignant from benign?* You'll find out by clustering the patients without ever showing K-Means the labels, and only then comparing the clusters with the true diagnoses.

> 🧰 **New tools for this challenge**
>
> * **Majority-label mapping:** K-Means numbers its clusters 0 and 1 in no particular order. To compare clusters with diagnoses, label each cluster with the diagnosis that is **most common among its patients**, then score those labels with `accuracy_score` as usual. `pd.crosstab(clusters, y)` shows you which diagnosis dominates each cluster.
> * **`homogeneity_score(y, clusters)`**: "Is each cluster pure?" It is 1.0 when every cluster contains patients with only one diagnosis.
> * **`completeness_score(y, clusters)`**: "Is each diagnosis kept together?" It is 1.0 when all patients with the same diagnosis land in the same cluster.
>
> Both are already imported in the Setup cell, and both take the true labels first and the cluster labels second.

**Your Task:**
1. **Reduce**: Fit `PCA(n_components=2, random_state=RANDOM_STATE)` on `X_scaled` (all 569 patients, no split) and transform it to 2 components.
2. **Cluster**: Fit `KMeans(n_clusters=2, n_init=10, random_state=RANDOM_STATE)` on the 2 components. 📝 Record the silhouette score of the clusters on the 2-component data (4 decimals).
3. **Compare with the truth**: Map each cluster to its majority diagnosis. 📝 Record the accuracy of those mapped labels against `y` (4 decimals).
4. **Score the clusters**: 📝 Record the homogeneity and completeness scores (4 decimals). Which one is higher, and what does that tell you about the clusters?
5. **Visualize and reflect**: Make two scatter plots of PC1 against PC2, one colored by cluster and one by the true diagnosis. How does this unsupervised accuracy compare with Challenge 3's supervised models, and why would you still use the labels when you have them?

In [ ]:
# Your code here


*Steps 4–5 — Your answers:*

---

## 🎓 Lab Wrap-Up & Reflection

### ✅ What You Accomplished
In this lab, you practiced:
- Scaling data and fitting K-Means, then choosing K with the elbow and silhouette scores
- Profiling clusters and naming them as segments a business can act on
- Running the PCA workflow and reading scree plots and loadings
- Comparing a model on all features with one on a few PCA components
- Clustering on PCA components and checking the clusters against labels K-Means never saw

### 🤔 Reflection Questions
Take 2–3 minutes to consider:
- In Challenge 1, would a real estate team prefer 3 segments or more? What would you need to know to decide?
- In Challenge 3, PCA cut the number of inputs sharply. When would that trade be worth a small loss in accuracy, and when wouldn't it?
- Challenge 4 found much of the diagnosis structure with no labels at all. Where in business might "structure without labels" be valuable?

### 🔗 Connection to Course Goals
Supervised learning needs a target someone has already recorded. Unsupervised learning works before that target exists: finding segments to target, spotting unusual records, or shrinking data so it can be seen and modeled. It's often the first step of a project, not the last.

### 📋 Next Steps
- **Homework:** Complete the Canvas quiz using your numerical results from today's challenges, and upload your completed notebook (due Sunday)
- **Next Tuesday:** Modern machine learning algorithms and a roadmap for what to learn next
- **Additional Practice:** Try the end-of-chapter exercises in Chapters 31–32

---
**💾 Save your work.** You'll upload this notebook with your quiz, so make sure every cell has been run and shows its output.

## 🚨 Troubleshooting & Common Issues

**Issue 1:** My inertia or silhouette scores don't match my groupmates'
- **Solution:** Check `n_init=10` and `random_state=RANDOM_STATE` in every `KMeans`, and that you scaled the features first. In Challenge 1, make sure you filled the missing `TotalBsmtSF` and `GarageArea` values with 0 before scaling.

**Issue 2:** My cluster numbers are in a different order than my groupmates'
- **Solution:** That's normal. Cluster numbers are arbitrary names. Compare cluster **profiles** (average feature values), not the numbers themselves.

**Issue 3:** My PCA variance percentages are different
- **Solution:** Standardize the features before PCA, and fit PCA on the standardized data. Unscaled data gives very different (and misleading) results.

**Issue 4:** My logistic regression accuracies don't match
- **Solution:** Check `test_size=0.2`, `random_state=RANDOM_STATE` and `stratify=y` in the split; `max_iter=10000` and `random_state=RANDOM_STATE` in `LogisticRegression`; and `random_state=RANDOM_STATE` in `PCA`. Fit the scaler and PCA on the **training** data only, then `transform()` the test data.

**Issue 5:** In Challenge 4, my accuracy is very low (below 0.5)
- **Solution:** Your clusters are probably numbered the opposite way from the diagnoses. Map each cluster to its **majority** diagnosis first (`pd.crosstab(clusters, y)` shows it), then compute accuracy on the mapped labels.

**Issue 6:** `homogeneity_score` and `completeness_score` look swapped
- **Solution:** Both take the **true labels first** and the cluster labels second: `homogeneity_score(y, clusters)`.

**General Debugging Tips:**
- Print shapes after each step to check that the data is what you expect
- If a variable seems wrong, re-run the notebook from the top (Runtime → Run all) to clear out stale values